# IFRS 9 ECL Calculation Engine

Combines PD, LGD, EAD, and stage classification into the final Expected Credit Loss (ECL)
calculation. Stage 1 loans use 12-month PD; Stage 2 and Stage 3 loans use a lifetime PD
derived from the 12-month PD using the standard survival-based approximation:
lifetime_PD = 1 - (1 - PD_12month)^n, where n is years to loan maturity.

In [1]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv('../data/processed/staged_loan_book.csv')
lgd_df = pd.read_csv('../data/processed/lgd_estimates.csv')
ead_df = pd.read_csv('../data/processed/ead_estimates.csv')

df = df.merge(lgd_df[['loan_id', 'LGD']], on='loan_id')
df = df.merge(ead_df[['loan_id', 'EAD']], on='loan_id')

df.shape

(10000, 20)

In [2]:
df['balance_to_original_ratio'] = (df['current_balance'] / df['original_balance']).round(3)
df['high_utilization_flag'] = np.where(df['utilization'] > 0.8, 1, 0)
df['score_deteriorated_flag'] = np.where(df['credit_score_change'] < -30, 1, 0)

df_encoded = pd.get_dummies(df, columns=['loan_type', 'region', 'loan_purpose'], drop_first=True)

print(list(df_encoded.columns))

['loan_id', 'origination_date', 'orig_credit_score', 'current_credit_score', 'original_balance', 'current_balance', 'credit_limit', 'interest_rate', 'days_past_due', 'watchlist_flag', 'default_flag', 'months_on_book', 'utilization', 'credit_score_change', 'stage', 'LGD', 'EAD', 'balance_to_original_ratio', 'high_utilization_flag', 'score_deteriorated_flag', 'loan_type_term', 'region_North', 'region_South', 'region_West', 'loan_purpose_debt_consolidation', 'loan_purpose_home_improvement', 'loan_purpose_personal']


In [3]:
model = joblib.load('../src/pd_model.pkl')
scaler = joblib.load('../src/pd_scaler.pkl')

In [4]:
model_features = [
    'orig_credit_score', 'current_credit_score', 'credit_score_change',
    'original_balance', 'current_balance', 'balance_to_original_ratio',
    'interest_rate', 'months_on_book', 'days_past_due',
    'watchlist_flag', 'high_utilization_flag', 'score_deteriorated_flag',
    'loan_type_term', 'region_North', 'region_South', 'region_West',
    'loan_purpose_debt_consolidation', 'loan_purpose_home_improvement', 'loan_purpose_personal'
]

X_new = df_encoded[model_features]
X_new_scaled = pd.DataFrame(scaler.transform(X_new), columns=X_new.columns, index=X_new.index)

df_encoded['PD_12month'] = model.predict_proba(X_new_scaled)[:, 1]

df_encoded[['loan_id', 'PD_12month']].head()

,loan_id,PD_12month
0,1,0.236738
1,2,0.150967
2,3,0.022888
3,4,0.054100
4,5,0.103698


In [5]:
df_encoded['years_remaining'] = np.clip(5 - (df_encoded['months_on_book'] / 12), 1, 5)

df_encoded['PD_lifetime'] = 1 - (1 - df_encoded['PD_12month']) ** df_encoded['years_remaining']

In [6]:
df_encoded['PD_used'] = np.where(
    df_encoded['stage'] == 'Stage 1',
    df_encoded['PD_12month'],
    df_encoded['PD_lifetime']
)

In [7]:
df_encoded['ECL'] = (df_encoded['PD_used'] * df_encoded['LGD'] * df_encoded['EAD']).round(2)

df_encoded[['loan_id', 'stage', 'PD_used', 'LGD', 'EAD', 'ECL']].head(10)

,loan_id,stage,PD_used,LGD,EAD,ECL
0,1,Stage 2,0.590886,0.687,1410.48,572.57
1,2,Stage 1,0.150967,0.692,10383.14,1084.72
2,3,Stage 1,0.022888,0.259,27026.64,160.21
3,4,Stage 1,0.054100,0.685,1598.14,59.22
4,5,Stage 1,0.103698,0.315,3752.12,122.56
5,6,Stage 1,0.066852,0.626,11193.45,468.44
6,7,Stage 1,0.068075,0.659,4537.56,203.56
7,8,Stage 1,0.027845,0.671,19855.53,370.99
8,9,Stage 2,0.488598,0.414,16035.77,3243.71
9,10,Stage 1,0.046857,0.471,3978.36,87.80


In [8]:
total_ecl = df_encoded['ECL'].sum()
total_exposure = df_encoded['EAD'].sum()
coverage_ratio = total_ecl / total_exposure

print(f"Total Portfolio ECL: ${total_ecl:,.2f}")
print(f"Total Exposure (EAD): ${total_exposure:,.2f}")
print(f"Coverage Ratio: {coverage_ratio:.2%}")

df_encoded.groupby('stage')['ECL'].agg(['sum', 'mean', 'count'])

Total Portfolio ECL: $7,924,494.99
Total Exposure (EAD): $99,081,601.80
Coverage Ratio: 8.00%


,sum,mean,count
stage,,,
Stage 1,1999034.28,274.103151,7293
Stage 2,5853954.15,2177.810324,2688
Stage 3,71506.56,3763.503158,19


In [9]:
df_encoded.to_csv('../data/processed/ecl_results.csv', index=False)
print("Saved.")

Saved.


In [10]:
print(df_encoded['PD_12month'].describe())

count    10000.000000
mean         0.087608
std          0.096266
min          0.003773
25%          0.027986
50%          0.054288
75%          0.109532
max          0.894583
Name: PD_12month, dtype: float64


In [11]:
print(df_encoded.groupby('stage')['PD_12month'].mean())

stage
Stage 1    0.053269
Stage 2    0.175962
Stage 3    0.768628
Name: PD_12month, dtype: float64


In [12]:
print(df_encoded['PD_lifetime'].describe())

count    10000.000000
mean         0.210236
std          0.195688
min          0.005327
25%          0.069528
50%          0.140561
75%          0.285255
max          0.999953
Name: PD_lifetime, dtype: float64
